## Benchmark of latents

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_benchmark 
# python -m ipykernel install --user --name scrna_cartography_py_benchmark --display-name "py_benchmark"

Load libraries

In [1]:
# Path-related libraries
import glob
import os
import sys
from pyhere import here  # For reproducible relative paths
from pathlib import Path

# AnnData and single-cell analysis libraries
import scanpy as sc       # For preprocessing and visualization of single-cell data
import anndata as ad      # For handling AnnData objects

# Numerical and tensor operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd
import warnings

# Benchmark
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
from scib_metrics.nearest_neighbors import NeighborsResults
from cuvs.neighbors import cagra
import cupy as cp

/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def cagra_nn(X: np.ndarray, k: int):
    # Setup data
    X = cp.asarray(np.ascontiguousarray(X, dtype=np.float32))

    # Build index
    index_params = cagra.IndexParams(graph_degree = 128)
    index = cagra.build(index_params, X)

    # Search
    search_params = cagra.SearchParams(itopk_size = 128)
    distances, neighbors = cagra.search(search_params, index, X, k)
    
    # Clean and return
    del index
    return NeighborsResults(indices=cp.asnumpy(neighbors), distances=np.sqrt(cp.asnumpy(distances)))

Parameters

In [3]:
# Saving
base_dir = str(here('data/integrate/third_pass/'))
file_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plot') 
tmp_dir = os.path.join(base_dir, 'tmp') 
model_dir = os.path.join(base_dir, 'models') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata')) 

In [8]:
# hvgs 
hvgs = [1000, 2000, 4000]

technical_seperate = ['cell_nuclei', 'ic_id_study', 'library_prep', 'ic_id_donor_overall']

celltype = 'study_cell_annotation_harmonized' 

Load adata object

In [5]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AF_combined.h5ad"))

Embeddings to test

In [6]:
embedding_keys = [x for x in adata.obsm.keys()]

Subset to annotated cells

In [7]:
celltype = "study_cell_annotation_harmonized"

n_before = adata.n_obs
adata_sub = adata[~adata.obs[celltype].isin(["unknown", "excluded"])].copy()
adata_sub.obs[celltype] = adata_sub.obs[celltype].astype("category")
adata_sub.obs[celltype] = adata_sub.obs[celltype].cat.remove_unused_categories()

n_after = adata_sub.n_obs
print(f"Filtered {n_before - n_after} cells (kept {n_after})")

Filtered 263180 cells (kept 239683)


In [11]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=FutureWarning,
        message="Argument `use_highly_variable` is deprecated.*"
    )
    
    results = {}
    results_norm = {}
    
    for technical in technical_seperate:

        print(f'Running {technical}')
        
        bm = Benchmarker(
            adata_sub,
            batch_key=technical,
            label_key=celltype,
            bio_conservation_metrics=BioConservation(),
            batch_correction_metrics=BatchCorrection(kbet_per_label = False),
            embedding_obsm_keys=embedding_keys,
            n_jobs=-1,
        )
        bm.prepare(neighbor_computer=cagra_nn)
        bm.benchmark()

        # save results
        results[technical] = bm.get_results(min_max_scale=False)
        results_norm[technical] = bm.get_results(min_max_scale=True)

        bm.get_results(min_max_scale=False).to_csv(os.path.join(file_dir, f'benchmark_celltype_sub_results_{technical}.csv'), index=True)
        bm.get_results(min_max_scale=True).to_csv(os.path.join(file_dir, f'benchmark_celltype_sub_norm_results_{technical}.csv'), index=True)
    
    
        print(f"[OK] Finished benchmark for {technical}")

Running cell_nuclei


Metrics:  60%|██████    | 6/10 [00:30<00:12,  3.07s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   4%|▎         | 1/27 [00:31<13:40, 31.54s/it]tch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [00:31<00:12,  3.13s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   7%|▋         |

[OK] Finished benchmark for cell_nuclei
Running ic_id_study


Metrics:  60%|██████    | 6/10 [00:47<00:28,  7.19s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   4%|▎         | 1/27 [00:49<21:18, 49.17s/it]tch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [00:32<00:13,  3.44s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   7%|▋         |

[OK] Finished benchmark for ic_id_study
Running library_prep


Metrics:  60%|██████    | 6/10 [00:40<00:20,  5.13s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   4%|▎         | 1/27 [00:41<18:07, 41.82s/it]tch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [00:33<00:14,  3.59s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   7%|▋         |

[OK] Finished benchmark for library_prep
Running ic_id_donor_overall


Metrics:  60%|██████    | 6/10 [00:43<00:23,  5.90s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   4%|▎         | 1/27 [00:45<19:31, 45.07s/it]tch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [00:32<00:13,  3.28s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   7%|▋         |

[OK] Finished benchmark for ic_id_donor_overall


/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/benchmark/_core.py:307: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Aggregate score' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[_METRIC_TYPE, per_class_score.columns] = _AGGREGATE_SCORE
/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/benchmark/_core.py:307: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Aggregate score' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[_METRIC_TYPE, per_class_score.columns] = _AGGREGATE_SCORE
/work/islet_cartography_scrna/scrna_cartography_py_benchmark/lib/python3.13/site-packages/scib_metrics/benchmark/_core.py:307: FutureWarning: 

Load results table and combine into one dataframe

In [11]:
# Get path to results (not normalised)
result_path = glob.glob(os.path.join(file_dir, "*.csv"))
result_path = [x for x in result_path if "norm" not in x]

df_list = []
for res in result_path:
    # Get base name
    base_name = Path(res).stem.replace("benchmark_celltype_sub_results_", "")
    # Load file
    df_tmp = pd.read_csv(res, index_col=0)
    # Transpose, add variable name and pivot longer
    df_tmp = df_tmp.transpose()
    # Make numeric
    cols = embedding_keys
    df_tmp[cols] = df_tmp[cols].apply(pd.to_numeric, errors='coerce')
    
    # Drop cluster dependet metrics and precalculated aggregate scores
    # As we are not going to cluster based on the KNN, and as these labels are from studies that have not been corrected
    df_tmp = df_tmp.drop(['KMeans NMI', 'KMeans ARI', 'Total', 'Batch correction', 'Bio conservation'])

    # Calculate aggregate scores
    df_means = df_tmp.groupby('Metric Type').mean(numeric_only=True)
    df_means = df_means.transpose()
    df_means['Total'] = 0.5 * df_means["Batch correction"] + 0.5 * df_means["Bio conservation"]
    df_means = df_means.transpose()
    
    # Prepare for combining with df_tmp
    df_means.index.name = 'Embedding'  # to align with df_tmp index
    df_means['Metric Type'] = 'Aggregate score'

    # combine with tmp
    df_tmp = pd.concat([df_tmp, df_means])
    df_tmp["Variable"] = base_name
    df_tmp = df_tmp.rename_axis("Metric").reset_index()
    df_tmp = df_tmp.melt(
        id_vars=["Metric", "Metric Type", "Variable"],
        var_name="Latent",
        value_name="Score",
    )

    # Add to list
    df_list.append(df_tmp)

# Combine results to one df
df_res = pd.concat(df_list)

# round to 4 digits
df_res = df_res.round(4)
print(df_res)

# Save results
df_res.to_csv(os.path.join(file_dir, 'benchmark_results_per_variable.csv'))

                 Metric       Metric Type             Variable      Latent  \
0       Isolated labels  Bio conservation  ic_id_donor_overall  X_latent_0   
1      Silhouette label  Bio conservation  ic_id_donor_overall  X_latent_0   
2                 cLISI  Bio conservation  ic_id_donor_overall  X_latent_0   
3                  BRAS  Batch correction  ic_id_donor_overall  X_latent_0   
4                 iLISI  Batch correction  ic_id_donor_overall  X_latent_0   
..                  ...               ...                  ...         ...   
265  Graph connectivity  Batch correction          cell_nuclei  X_latent_9   
266      PCR comparison  Batch correction          cell_nuclei  X_latent_9   
267    Batch correction   Aggregate score          cell_nuclei  X_latent_9   
268    Bio conservation   Aggregate score          cell_nuclei  X_latent_9   
269               Total   Aggregate score          cell_nuclei  X_latent_9   

      Score  
0    0.5032  
1    0.6243  
2    1.0000  
3    0.

Calculat mean aggregate across all variables

In [13]:
df_res

,Metric,Metric Type,Variable,Latent,Score
0,Isolated labels,Bio conservation,ic_id_donor_overall,X_latent_0,0.5032
1,Silhouette label,Bio conservation,ic_id_donor_overall,X_latent_0,0.6243
2,cLISI,Bio conservation,ic_id_donor_overall,X_latent_0,1.0000
3,BRAS,Batch correction,ic_id_donor_overall,X_latent_0,0.7942
4,iLISI,Batch correction,ic_id_donor_overall,X_latent_0,0.0378
...,...,...,...,...,...
265,Graph connectivity,Batch correction,cell_nuclei,X_latent_9,0.7533
266,PCR comparison,Batch correction,cell_nuclei,X_latent_9,0.9672
267,Batch correction,Aggregate score,cell_nuclei,X_latent_9,0.6697
268,Bio conservation,Aggregate score,cell_nuclei,X_latent_9,0.7295


In [12]:
# Mean of aggregated scores
df_total = df_res[
    (df_res["Metric"].isin(["Bio conservation", "Batch correction", "Total"]))
].groupby(['Latent','Metric']).mean(numeric_only=True).round(4)

print(df_total)
df_total.to_csv(os.path.join(file_dir,'benchmark_results_total.csv'))

                              Score
Latent     Metric                  
X_latent_0 Batch correction  0.6548
           Bio conservation  0.7148
           Total             0.6848
X_latent_1 Batch correction  0.6590
           Bio conservation  0.7150
...                             ...
X_latent_8 Bio conservation  0.7053
           Total             0.6819
X_latent_9 Batch correction  0.6578
           Bio conservation  0.7123
           Total             0.6851

[81 rows x 1 columns]
